# **Librerias**

In [56]:
# CARGAR LIBRERIAS NECESARIAS
import warnings
import re
import unicodedata
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
!pip install pyreadstat
import pyreadstat
from IPython.display import display, HTML

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as mtick

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    roc_curve
)
from sklearn.preprocessing import OneHotEncoder

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression

# CatBoost
!pip install catboost
from catboost import CatBoostClassifier, Pool
!pip install shap
import shap


# **Carga Datasets**

In [57]:
# CARGAR DATASETS: BBDD ENCUESTA NACIONAL DE ACCESO Y USOS DE INTERNET SUBTEL (https://www.subtel.gob.cl/estudios/internet-y-sociedad-de-la-informacion/)

df2012 = pd.read_excel('/Users/tomas/GitHub/eaui_subtel/data/xlsx/2012.xlsx')
df2013 = pd.read_excel('/Users/tomas/GitHub/eaui_subtel/data/xlsx/2013.xlsx')
df2015 = pd.read_excel('/Users/tomas/GitHub/eaui_subtel/data/xlsx/2015.xlsx')
df2016 = pd.read_excel('/Users/tomas/GitHub/eaui_subtel/data/xlsx/2016.xlsx')
df2017 = pd.read_excel('/Users/tomas/GitHub/eaui_subtel/data/xlsx/2017.xlsx')
df2018 = pd.read_excel('/Users/tomas/GitHub/eaui_subtel/data/xlsx/2018.xlsx')
df2024 = pd.read_excel('/Users/tomas/GitHub/eaui_subtel/data/xlsx/2024.xlsx')
df2025 = pd.read_excel('/Users/tomas/GitHub/eaui_subtel/data/xlsx/2025.xlsx')

In [108]:
pd.options.display.float_format = '{:,.2f}'.format
pd.set_option('display.max_columns', None)

In [110]:
print(df2024.columns.tolist())

['numero_de_registro_variable_del_sistema', 'fecha_de_inicio_de_la_entrevista_variable_del_sistema', 'fecha_de_fin_de_la_entrevista_variable_del_sistema', 'exp_hog', 'region', 'comuna', 'zona', 'num_hogar', 'registre_numero_del_total_de_integrantes_del_hogar', 'a2_edad_miembro_1', 'a3_sexo_miembro_1', 'a5_registro_jefe_de_hogar_miembro_1', 'a2_edad_miembro_2', 'a3_sexo_miembro_2', 'a5_registro_jefe_de_hogar_miembro_2', 'a2_edad_miembro_3', 'a3_sexo_miembro_3', 'a5_registro_jefe_de_hogar_miembro_3', 'a2_edad_miembro_4', 'a3_sexo_miembro_4', 'a5_registro_jefe_de_hogar_miembro_4', 'a2_edad_miembro_5', 'a3_sexo_miembro_5', 'a5_registro_jefe_de_hogar_miembro_5', 'a2_edad_miembro_6', 'a3_sexo_miembro_6', 'a5_registro_jefe_de_hogar_miembro_6', 'a2_edad_miembro_7', 'a3_sexo_miembro_7', 'a5_registro_jefe_de_hogar_miembro_7', 'a2_edad_miembro_8', 'a3_sexo_miembro_8', 'a5_registro_jefe_de_hogar_miembro_8', 'a2_edad_miembro_9', 'a3_sexo_miembro_9', 'a5_registro_jefe_de_hogar_miembro_9', 'a2_edad_m

# **Recodificación de variables**

In [59]:
# ETIQUETAR DATOS SEGÚN AÑO DE LA ENCUESTA
df2012['anno'] = 2012
df2013['anno'] = 2013
df2015['anno'] = 2015
df2016['anno'] = 2016
df2017['anno'] = 2017
df2018['anno'] = 2018
df2024['anno'] = 2024
df2025['anno'] = 2025

# LIMPIAR DATOS (TILDES, MINUSCULAS, ESPACIOS, ETC)
def limpiar_df(df):

    nuevas_cols = []
    for col in df.columns:
        col_clean = unicodedata.normalize("NFKD", col).encode("ascii", "ignore").decode("utf-8")
        col_clean = col_clean.lower()
        col_clean = re.sub(r"[^a-z0-9]+", "_", col_clean)
        col_clean = re.sub(r"_+", "_", col_clean)
        col_clean = col_clean.strip("_")
        nuevas_cols.append(col_clean)

    df.columns = nuevas_cols

    def limpiar_texto(x):
        if isinstance(x, str):
            x = unicodedata.normalize("NFKD", x).encode("ascii", "ignore").decode("utf-8")
            x = x.lower()
            x = re.sub(r"\s+", " ", x)
            x = x.strip()

            return x
        return x

    df = df.applymap(limpiar_texto)

    return df

# LIMPIEZA DE DATOS
limpiar_df(df2012)
limpiar_df(df2013)
limpiar_df(df2015)
limpiar_df(df2016)
limpiar_df(df2017)
limpiar_df(df2018)
limpiar_df(df2024)
limpiar_df(df2025)


,numero_de_registro,exp_hog,region,comuna,zona,las_primeras_preguntas_estan_relacionadas_con_informacion_sobre_el_uso_de_internet_la_persona_seleccionada_podra_responder_estas_preguntas,a9_cual_es_su_parentesco_con_el_jefe_a_de_hogar,a11_pensando_en_el_jefe_de_hogar_cual_fue_el_ultimo_curso_y_tipo_de_estudio_aprobado_por_el_jefe_de_hogar,a12_y_cual_es_la_profesion_o_trabajo_o_actividad_del_principal_sostenedor_del_hogar,aymara_a13_algun_miembro_de_este_hogar_pertenece_o_es_descendiente_de_alguno_de_estos_pueblos_indigenas_rm,...,q42_1_cuanto_es_lo_maximo_que_estaria_dispuesto_a_pagar_por_un_servicio_de_conexion_a_internet_movil_en_el_hogar,a12_1_a_cual_de_los_siguientes_tramos_corresponde_el_promedio_total_de_los_ingresos_familiares,unnamed_368,unnamed_369,unnamed_370,tipo_de_acceso_a_internet,ponderador_de_hogar,ponderador_de_personas,factor_de_expansion_personas,anno
0,3,478.65,13.- region metropolitana de santiago,lampa,rural,si,yo soy el jefe de hogar,superior tecnica completa,"4.empleado administrativo medio y bajo, vended...",0,...,20000,"$ 850 mil a $ 1,4 millones",34,hombre,c2,fija + movil,0.40,0.65,"1,916.04",2025
1,4,478.65,13.- region metropolitana de santiago,lampa,rural,si,yo soy el jefe de hogar,media cientifico-humanista incompleta,1.trabajos menores ocasionales e informales (l...,0,...,5000,$ 280 mil a $ 487 mil,59,hombre,d,fija + movil,0.40,0.41,"1,212.70",2025
2,5,478.65,13.- region metropolitana de santiago,lampa,rural,si,hijo(a),basica completa,1.trabajos menores ocasionales e informales (l...,0,...,10000,$ 280 mil a $ 487 mil,27,mujer,e,fija + movil,0.40,0.47,"1,378.27",2025
3,8,"5,688.41",13.- region metropolitana de santiago,penalolen,urbana,si,yo soy el jefe de hogar,superior universitaria completa,"4.empleado administrativo medio y bajo, vended...",0,...,9000,$ 394 mil a $ 686 mil,75,hombre,c2,solo movil,4.73,4.32,"12,686.11",2025
4,9,"5,688.41",13.- region metropolitana de santiago,santiago,urbana,si,hijo(a),basica incompleta,"5.ejecutivo medio (gerente, subgerente). geren...",0,...,10000,$ 211 mil a $ 366 mil,23,mujer,c3,solo movil,4.73,6.69,"19,643.96",2025
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4695,6017,364.60,11.- region de aysen,coyhaique,rural,si,yo soy el jefe de hogar,media cientifico-humanista completa,"4.empleado administrativo medio y bajo, vended...",0,...,20000,$ 394 mil a $ 686 mil,20,hombre,c3,solo movil,0.30,0.54,"1,590.25",2025
4696,6018,341.46,11.- region de aysen,coyhaique,urbana,si,conyuge o pareja,media cientifico-humanista completa,"3.obrero calificado, capataz, junior microempr...",0,...,10000,"$ 640 mil a $ 1,1 millones",19,mujer,c3,fija + movil,0.28,0.37,"1,095.45",2025
4697,6022,364.60,11.- region de aysen,coyhaique,rural,si,conyuge o pareja,media tecnico-profesional completa,"3.obrero calificado, capataz, junior microempr...",0,...,6000,$ 367 mil a $ 639 mil,57,mujer,d,solo movil,0.30,0.18,531.41,2025
4698,6023,364.60,11.- region de aysen,coyhaique,rural,si,conyuge o pareja,media tecnico-profesional incompleta,"2.oficio menor, obrero no calificado, jornaler...",0,...,22900,nsnr,47,mujer,d,solo movil,0.30,0.26,767.73,2025


In [60]:
# NORMALIZAR NOMBRES DE REGIONES
REGIONES_BASE = {
    "1":  "tarapaca",
    "2":  "antofagasta",
    "3":  "atacama",
    "4":  "coquimbo",
    "5":  "valparaiso",
    "6":  "ohiggins",
    "7":  "maule",
    "8":  "biobio",
    "9":  "araucania",
    "10": "los_lagos",
    "11": "aysen",
    "12": "magallanes",
    "13": "metropolitana",
    "14": "los_rios",
    "15": "arica_y_parinacota",
    "16": "nuble"
}

REGIONES_LABEL = {k: f"{k} {v}" for k, v in REGIONES_BASE.items()}

def limpiar_texto(texto):
    if not isinstance(texto, str):
        return texto
    texto = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode("utf-8")
    return texto.lower().strip()

def normalizar_region(df, col="region"):
    def procesar(valor):
        if pd.isna(valor):
            return valor

        txt = limpiar_texto(str(valor))

        m = re.search(r"\b(1[0-6]|[1-9])\b", txt)
        if m:
            num = m.group(1)
            return REGIONES_LABEL[num]

        txt_no_space = txt.replace(" ", "")
        for num, nombre_base in REGIONES_BASE.items():
            nombre_txt = nombre_base.replace("_", " ")
            nombre_no_space = nombre_txt.replace(" ", "")
            if nombre_txt in txt or nombre_no_space in txt_no_space:
                return REGIONES_LABEL[num]
        return txt

    df = df.copy()
    df[col] = df[col].apply(procesar)

    ordered_categories = [REGIONES_LABEL[str(i)] for i in range(1, 17)]
    df[col] = pd.Categorical(df[col], categories=ordered_categories, ordered=True)

    return df

df2012 = normalizar_region(df2012, "region")
df2013 = normalizar_region(df2013, "region")
df2015 = normalizar_region(df2015, "region")
df2016 = normalizar_region(df2016, "region")
df2017 = normalizar_region(df2017, "region")
df2018 = normalizar_region(df2018, "region")
df2024 = normalizar_region(df2024, "region")
df2025 = normalizar_region(df2025, "region")

In [61]:
# RENOMBRADO DE VARIABLES DE ACCESO A INTERNET
df2012["acceso_int"] = df2012["p5_los_miembros_de_este_hogar_tienen_acceso_a_internet_desde_el_hogar_sin_imp"]
df2013["acceso_int"] = df2013["los_miembros_de_este_hogar_tienen_acceso_a_internet_desde_el_hogar"]
df2015["acceso_int"] = df2015["p5_los_miembros_de_este_hogar_tienen_acceso_pagado_y_propio_a_internet_desd"]
df2016["acceso_int"] = df2016["p5_los_miembros_de_este_hogar_tienen_acceso_pagado_y_propio_a_internet_desde_el_hogar_sin_importar_si_lo_utilizan_o_no"]
df2017["acceso_int"] = df2017["p5_los_miembros_de_este_hogar_tienen_acceso_pagado_y_propio_a_internet_desde_el_hogar_sin_importar_si_lo_utilizan_o_no"]
df2018["acceso_int"] = df2018["p5_el_acceso_a_internet_puede_ser_con_internet_fijo_o_movil_via_computador_telefono_movil_o_smartphone_tablet_tv_o_consola_de_juegos_con_acceso_a_internet_habilitado_los_miembros_de_este_hogar_tienen_acceso_propio_y_pagado_a_internet_desde_el_hogar"]
df2024["acceso_int"] = df2024["p1_los_miembros_de_este_hogar_tienen_acceso_propio_y_pagado_a_internet_desde_el_hogar_sin_importar_si_lo_utilizan_o_no"]
df2025["acceso_int"] = df2025["p1_los_miembros_de_este_hogar_tienen_acceso_propio_y_pagado_a_internet_desde_el_hogar_sin_importar_si_lo_utilizan_o_no"]


In [62]:
def normalizar_acceso_int(df, col="acceso_int"):
    def limpiar(x):
        if pd.isna(x):
            return x
        x = str(x)

        x = unicodedata.normalize("NFKD", x).encode("ascii", "ignore").decode("utf-8")
        x = x.lower()
        x = re.sub(r"^\s*\d+\s*[.\-]*\s*", "", x)
        x = x.strip()
        x_sin_espacios = x.replace(" ", "")

        if x_sin_espacios in ["si", "s"]:
            return "si"

        if x_sin_espacios in ["no", "n"]:
            return "no"

        return pd.NA

    df[col] = df[col].apply(limpiar)
    return df

normalizar_acceso_int(df2012)
normalizar_acceso_int(df2013)
normalizar_acceso_int(df2015)
normalizar_acceso_int(df2016)
normalizar_acceso_int(df2017)
normalizar_acceso_int(df2018)
normalizar_acceso_int(df2024)
normalizar_acceso_int(df2025)

df2017.head()

,sbjnum,exp_hog,zona,region,comuna,p5_los_miembros_de_este_hogar_tienen_acceso_pagado_y_propio_a_internet_desde_el_hogar_sin_importar_si_lo_utilizan_o_no,computador_fijo_pc_desktop,computador_portatil_notebook_laptop_o_netbook,tablet_ipad_galaxy_tab_xperia_tablet_hp_palm_touchpad_etc,telefono_movil_o_smartphone_iphone_galaxy_palm_blackberry_xperia_etc,...,sexo_del_jefe_de_hogar,edad_del_jefe_de_hogar,estado_civil_del_jefe_de_hogar,educacion_del_jefe_de_hogar,etnia_del_jefe_de_hogar,discapacidad_del_jefe_de_hogar,conexion_hogares,p16_1_entre_las_razones_senaladas_indique_cual_considera_la_mas_importante_ru,anno,acceso_int
0,39712169,4547,Urbana,13 metropolitana,PROVIDENCIA,SI,No,Yes,No,Yes,...,Hombre,33,6. Soltero(a),11. Sup. Universitaria Completa,10. No pertenece a ningún pueblo indígena en C...,7. No tiene ninguna condición de larga duración,Fija + Móvil,NaN,2017,si
1,39714777,4547,Urbana,13 metropolitana,PROVIDENCIA,SI,No,Yes,Yes,Yes,...,Hombre,38,6. Soltero(a),10. Sup. Universitaria Incompleta,10. No pertenece a ningún pueblo indígena en C...,7. No tiene ninguna condición de larga duración,Sólo Fija,NaN,2017,si
2,39716085,4547,Urbana,13 metropolitana,PROVIDENCIA,SI,No,Yes,Yes,Yes,...,Hombre,75,1. Casado/a,11. Sup. Universitaria Completa,10. No pertenece a ningún pueblo indígena en C...,7. No tiene ninguna condición de larga duración,Sólo Fija,NaN,2017,si
3,39717252,4547,Urbana,13 metropolitana,PROVIDENCIA,SI,Yes,No,No,Yes,...,Mujer,40,6. Soltero(a),11. Sup. Universitaria Completa,10. No pertenece a ningún pueblo indígena en C...,7. No tiene ninguna condición de larga duración,Sólo Fija,NaN,2017,si
4,39718603,4547,Urbana,13 metropolitana,PROVIDENCIA,SI,Yes,Yes,No,Yes,...,Mujer,54,3. Anulado(a) / Divorciado(a),11. Sup. Universitaria Completa,10. No pertenece a ningún pueblo indígena en C...,7. No tiene ninguna condición de larga duración,Fija + Móvil,NaN,2017,si


In [63]:
def normalizar_zona(df, col="zona"):
    def limpiar(x):

        if pd.isna(x):
            return x

        if isinstance(x, (int, float)):
            if x == 1:
                return "urbano"
            if x == 2:
                return "rural"
            return pd.NA

        x = str(x)
        x = unicodedata.normalize("NFKD", x).encode("ascii", "ignore").decode("utf-8")
        x = x.lower()
        x = re.sub(r"^\s*\d+\s*[\.\-\)]*\s*", "", x)
        x = x.strip()
        x_sin_esp = x.replace(" ", "")
        if any(term in x_sin_esp for term in ["urbano", "urbana", "urbanos", "urbanas"]):
            return "urbano"

        if any(term in x_sin_esp for term in ["rural", "rurales"]):
            return "rural"

        if x_sin_esp == "1":
            return "urbano"
        if x_sin_esp == "2":
            return "rural"

        return pd.NA

    df[col] = df[col].apply(limpiar)
    return df

normalizar_zona(df2012)
normalizar_zona(df2013)
normalizar_zona(df2015)
normalizar_zona(df2016)
normalizar_zona(df2017)
normalizar_zona(df2018)
normalizar_zona(df2024)
normalizar_zona(df2025)

,numero_de_registro,exp_hog,region,comuna,zona,las_primeras_preguntas_estan_relacionadas_con_informacion_sobre_el_uso_de_internet_la_persona_seleccionada_podra_responder_estas_preguntas,a9_cual_es_su_parentesco_con_el_jefe_a_de_hogar,a11_pensando_en_el_jefe_de_hogar_cual_fue_el_ultimo_curso_y_tipo_de_estudio_aprobado_por_el_jefe_de_hogar,a12_y_cual_es_la_profesion_o_trabajo_o_actividad_del_principal_sostenedor_del_hogar,aymara_a13_algun_miembro_de_este_hogar_pertenece_o_es_descendiente_de_alguno_de_estos_pueblos_indigenas_rm,...,a12_1_a_cual_de_los_siguientes_tramos_corresponde_el_promedio_total_de_los_ingresos_familiares,unnamed_368,unnamed_369,unnamed_370,tipo_de_acceso_a_internet,ponderador_de_hogar,ponderador_de_personas,factor_de_expansion_personas,anno,acceso_int
0,3,478.65,13 metropolitana,LAMPA,rural,Si,Yo soy el jefe de Hogar,Superior Técnica Completa,"4.Empleado administrativo medio y bajo, vended...",0,...,"$ 850 mil a $ 1,4 millones",34,Hombre,C2,Fija + Móvil,0.40,0.65,"1,916.04",2025,si
1,4,478.65,13 metropolitana,LAMPA,rural,Si,Yo soy el jefe de Hogar,Media Científico-Humanista Incompleta,1.Trabajos menores ocasionales e informales (L...,0,...,$ 280 mil a $ 487 mil,59,Hombre,D,Fija + Móvil,0.40,0.41,"1,212.70",2025,si
2,5,478.65,13 metropolitana,LAMPA,rural,Si,Hijo(a),Básica completa,1.Trabajos menores ocasionales e informales (L...,0,...,$ 280 mil a $ 487 mil,27,Mujer,E,Fija + Móvil,0.40,0.47,"1,378.27",2025,si
3,8,"5,688.41",13 metropolitana,PEÑALOLÉN,urbano,Si,Yo soy el jefe de Hogar,Superior Universitaria Completa,"4.Empleado administrativo medio y bajo, vended...",0,...,$ 394 mil a $ 686 mil,75,Hombre,C2,Solo Móvil,4.73,4.32,"12,686.11",2025,si
4,9,"5,688.41",13 metropolitana,SANTIAGO,urbano,Si,Hijo(a),Básica incompleta,"5.Ejecutivo medio (gerente, subgerente). Geren...",0,...,$ 211 mil a $ 366 mil,23,Mujer,C3,Solo Móvil,4.73,6.69,"19,643.96",2025,si
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4695,6017,364.60,11 aysen,COYHAIQUE,rural,Si,Yo soy el jefe de Hogar,Media Científico-Humanista Completa,"4.Empleado administrativo medio y bajo, vended...",0,...,$ 394 mil a $ 686 mil,20,Hombre,C3,Solo Móvil,0.30,0.54,"1,590.25",2025,si
4696,6018,341.46,11 aysen,COYHAIQUE,urbano,Si,Cónyuge o Pareja,Media Científico-Humanista Completa,"3.Obrero calificado, capataz, Junior microempr...",0,...,"$ 640 mil a $ 1,1 millones",19,Mujer,C3,Fija + Móvil,0.28,0.37,"1,095.45",2025,si
4697,6022,364.60,11 aysen,COYHAIQUE,rural,Si,Cónyuge o Pareja,Media Técnico-Profesional Completa,"3.Obrero calificado, capataz, Junior microempr...",0,...,$ 367 mil a $ 639 mil,57,Mujer,D,Solo Móvil,0.30,0.18,531.41,2025,si
4698,6023,364.60,11 aysen,COYHAIQUE,rural,Si,Cónyuge o Pareja,Media Técnico-Profesional Incompleta,"2.Oficio menor, obrero no calificado, jornaler...",0,...,NSNR,47,Mujer,D,Solo Móvil,0.30,0.26,767.73,2025,si


# **Variables sociodemográficas y económicas**

Luego, para incluir en el modelo variables sociales, demográficas y económicas, se desarrollan las variables `ingreso_tot` (ingreso total del hogar), `quintil` (quintil de ingreso a partir de los datos de `ingreso_tot`), `niveduc` (nivel educacional del encuestado), `edad`, `edad_tramo` (tramos de edad de 5 años a partir de los 18 años) y `sexo`.

Al igual que en todas las demás variables trabajadas, se desarrollaron funciones que permiten la estandarización de los valores de respuesta de cada uno de los dataframes debido a sus discrepancias.

### **Ingreso del hogar**

In [64]:
#@title
# CALCULOS DE INGRESOS TOTALES

# año 2012
cols_p34 = [col for col in df2012.columns if col.startswith("p34_")]

df2012[cols_p34] = df2012[cols_p34].apply(
    lambda x: pd.to_numeric(x, errors="coerce")
)

df2012["ingreso_tot"] = df2012[cols_p34].sum(axis=1)

# año 2013
df2013['ingreso_tot'] = df2013['ingreso_autnomo_total_del_hogar']

# año 2015
df2015['ingreso_tot'] = df2015['ingreso_hogar']

# año 2016
df2016['ingreso_tot'] = df2016['ingrtot']

# año 2017
df2017['ingreso_tot'] = df2017['n_pers']* df2017['ing_pcap2']



### **Quintiles de ingreso**

In [65]:
#@title
# QUINTILES DE INGRESO
def agregar_quintil(df, col_ing="ingreso_tot", col_quintil="quintil"):
    """
    Calcula el quintil en base a la columna ingreso_tot.
    Crea una columna 'quintil' con valores Q1, Q2, Q3, Q4, Q5.
    """
    df[col_ing] = pd.to_numeric(df[col_ing], errors="coerce")

    df[col_quintil] = pd.qcut(
        df[col_ing],
        q=5,
        labels=["Q1", "Q2", "Q3", "Q4", "Q5"]
    )
    return df

df2012 = agregar_quintil(df2012)
df2013 = agregar_quintil(df2013)
df2015 = agregar_quintil(df2015)
df2016 = agregar_quintil(df2016)
df2017 = agregar_quintil(df2017)

### **Nivel educacional**

In [66]:
#@title
# Construcción de variable Educacion

# 2012
df2012['niveduc'] = df2012['h5b_tipo_de_ensenanza']

# 2013
df2013['niveduc'] = df2013['tipo_de_estudio_aprobado_entrevistado']

# 2015
df2015['niveduc'] = df2015['ultimo_curso_y_tipo_de_estudio_aprobado']

# 2016
df2016['niveduc'] = df2016['educacion_1']

#2017
df2017['niveduc'] = df2017['educacion_del_entrevistado']

#2018
df2018['niveduc'] = df2018['a10_cual_fue_el_ultimo_curso_y_tipo_de_estudio_aprobado_por_el_jefe_de_hogar']

# NORMALIZAR VALORES DE VARIABLE

def normalizar_niveduc(df, col="niveduc"):
    """
    Normaliza niveles educativos en la columna `niveduc` de un DataFrame.
    Devuelve categorías estandarizadas en minúsculas y con guiones bajos:

    - sin_educacion_formal
    - basica_incompleta
    - basica_completa
    - media_ch_incompleta
    - media_ch_completa
    - media_tp_incompleta
    - media_tp_completa
    - sup_tecnica_incompleta
    - sup_tecnica_completa
    - sup_universitaria_incompleta
    - sup_universitaria_completa

    Valores como '99' o 'NS/NR' se convierten en NaN.
    """

    def limpiar_texto(x):
        if pd.isna(x):
            return x
        x = str(x)

        x = unicodedata.normalize("NFKD", x).encode("ascii", "ignore").decode("utf-8")
        x = x.lower().strip()
        x = re.sub(r"^\s*\d+\s*[\.\-\)]*\s*", "", x)
        x = re.sub(r"\s+", " ", x)
        return x

    def mapear_niveduc(x):
        if pd.isna(x):
            return pd.NA

        x = limpiar_texto(x)

        if x in ["99", "ns/nr", "nsnr", "ns nr", "no sabe/no responde"]:
            return pd.NA

        # SIN EDUCACION
        if "sin educacion formal" in x:
            return "sin_educacion_formal"

        # BASICA
        if "basica" in x:
            if "incompleta" in x:
                return "basica_incompleta"
            if "completa" in x:
                return "basica_completa"

        # MEDIA CIENTIFICO-HUMANISTA (media c-h / cientifico-humanista)
        if "media" in x and (
            "c-h" in x or "cientifico" in x or "humanista" in x
        ):
            if "incompleta" in x:
                return "media_ch_incompleta"
            if "completa" in x:
                return "media_ch_completa"

        # MEDIA TECNICO-PROFESIONAL (media t-p / tecnico-profesional)
        if "media" in x and (
            "t-p" in x or "tecnico" in x or "profesional" in x
        ):
            if "incompleta" in x:
                return "media_tp_incompleta"
            if "completa" in x:
                return "media_tp_completa"

        # SUPERIOR / SUP.
        if "sup" in x or "superior" in x:
            # universitaria
            if "universitaria" in x:
                if "incompleta" in x:
                    return "sup_universitaria_incompleta"
                if "completa" in x:
                    return "sup_universitaria_completa"
            # tecnica
            if "tecnica" in x:
                if "incompleta" in x:
                    return "sup_tecnica_incompleta"
                if "completa" in x:
                    return "sup_tecnica_completa"

        return x

    df[col] = df[col].apply(mapear_niveduc)
    return df

df2012 = normalizar_niveduc(df2012, "niveduc")
df2013 = normalizar_niveduc(df2013, "niveduc")
df2015 = normalizar_niveduc(df2015, "niveduc")
df2016 = normalizar_niveduc(df2016, "niveduc")
df2017 = normalizar_niveduc(df2017, "niveduc")
df2018 = normalizar_niveduc(df2018, "niveduc")

### **Edad**

In [67]:
#@title
# CREACIÓN DE VARIABLE "edad" POR AÑO (2012–2018)

# 2012
if "h4_edad_del_entrevistado" in df2012.columns:
    df2012["edad"] = pd.to_numeric(
        df2012["h4_edad_del_entrevistado"], errors="coerce"
    )

# 2013
if "edad_entrevistado" in df2013.columns:
    df2013["edad"] = pd.to_numeric(
        df2013["edad_entrevistado"], errors="coerce"
    )

# 2015
if "Edad" in df2015.columns:
    df2015["edad"] = pd.to_numeric(
        df2015["Edad"], errors="coerce"
    )

# 2016
if "edad_entrevistado" in df2016.columns:
    df2016["edad"] = pd.to_numeric(
        df2016["edad_entrevistado"], errors="coerce"
    )

# 2017
if "edad_entrevistado" in df2017.columns:
    df2017["edad"] = pd.to_numeric(
        df2017["edad_entrevistado"], errors="coerce"
    )

# 2018
if "edad_usos" in map(str.lower, df2018.columns):
    col_edad_2018 = [c for c in df2018.columns if c.lower() == "edad_usos"][0]
    df2018["edad"] = pd.to_numeric(
        df2018[col_edad_2018], errors="coerce"
    )

# ELIMINAR MENORES DE 18 AÑOS
def filtrar_mayores_edad(df, col_edad="edad", edad_min=18):
    if col_edad not in df.columns:
      return df

    return df.loc[df[col_edad].notna() & (df[col_edad] >= edad_min)].copy()

df2012 = filtrar_mayores_edad(df2012)
df2013 = filtrar_mayores_edad(df2013)
df2015 = filtrar_mayores_edad(df2015)
df2016 = filtrar_mayores_edad(df2016)
df2017 = filtrar_mayores_edad(df2017)
df2018 = filtrar_mayores_edad(df2018)

print("Variable edad creada y filtrada (>=18). Resumen por año:\n")

for year, df in [
    (2012, df2012),
    (2013, df2013),
    (2015, df2015),
    (2016, df2016),
    (2017, df2017),
    (2018, df2018),
]:
    print(f"{year}:")
    if "edad" in df.columns:
        print(df["edad"].describe())
        print(f"N observaciones: {len(df)}")
    else:
        print("Variable 'edad' no creada")
    print("-" * 40)


Variable edad creada y filtrada (>=18). Resumen por año:

2012:
count   3,617.00
mean       44.61
std        15.88
min        18.00
25%        31.00
50%        44.00
75%        57.00
max        86.00
Name: edad, dtype: float64
N observaciones: 3617
----------------------------------------
2013:
count   8,682.00
mean       43.17
std        16.27
min        18.00
25%        29.00
50%        42.00
75%        56.00
max        89.00
Name: edad, dtype: float64
N observaciones: 8682
----------------------------------------
2015:
count   3,562.00
mean       48.53
std        17.16
min        18.00
25%        34.00
50%        49.00
75%        61.00
max        96.00
Name: edad, dtype: float64
N observaciones: 3562
----------------------------------------
2016:
count   3,471.00
mean       42.07
std        15.85
min        18.00
25%        28.00
50%        41.00
75%        55.00
max        75.00
Name: edad, dtype: float64
N observaciones: 3471
----------------------------------------
2017:
count   

### **Tramos de edad**

In [100]:
#@title
def crear_edad_tramo(df, col_edad="edad", col_out="edad_tramo", top_age=80, top_label="76+"):
    if col_edad not in df.columns:
      return df

    edad = pd.to_numeric(df[col_edad], errors="coerce")


    edges = [18, 21] + list(range(26, top_age + 6, 5))
    labels = ["18–20"]
    start = 21
    for end in range(25, top_age + 1, 5):
        labels.append(f"{start}–{end}")
        start += 5

    tramo = pd.cut(
        edad,
        bins=edges,
        right=False,
        labels=labels,
        include_lowest=True
    )

    if top_age is not None:
        tramo = tramo.astype("object")
        tramo.loc[edad > top_age] = top_label

    df[col_out] = tramo
    return df


df2012 = crear_edad_tramo(df2012)
df2013 = crear_edad_tramo(df2013)
df2015 = crear_edad_tramo(df2015)
df2016 = crear_edad_tramo(df2016)
df2017 = crear_edad_tramo(df2017)
df2018 = crear_edad_tramo(df2018)
df2024 = crear_edad_tramo(df2024)
df2025 = crear_edad_tramo(df2025)

for year, df in [(2012, df2012), (2013, df2013), (2015, df2015), (2016, df2016), (2017, df2017), (2018, df2018)]:
    print(f"\n{year} - edad_tramo:")
    if "edad_tramo" in df.columns:
        print(df["edad_tramo"].value_counts(dropna=False).sort_index())



2012 - edad_tramo:
edad_tramo
18–20    192
21–25    333
26–30    340
31–35    321
36–40    384
41–45    342
46–50    364
51–55    327
56–60    305
61–65    256
66–70    254
71–75    187
76+        7
76–80      5
Name: count, dtype: int64

2013 - edad_tramo:
edad_tramo
18–20     580
21–25    1042
26–30     850
31–35     790
36–40     843
41–45     759
46–50     814
51–55     765
56–60     697
61–65     536
66–70     556
71–75     431
76+        11
76–80       8
Name: count, dtype: int64

2015 - edad_tramo:
edad_tramo
18–20    123
21–25    295
26–30    277
31–35    285
36–40    281
41–45    249
46–50    382
51–55    350
56–60    383
61–65    285
66–70    248
71–75    200
76+       82
76–80    122
Name: count, dtype: int64

2016 - edad_tramo:
edad_tramo
18–20    285
21–25    357
26–30    401
31–35    350
36–40    321
41–45    307
46–50    315
51–55    329
56–60    278
61–65    213
66–70    174
71–75    141
Name: count, dtype: int64

2017 - edad_tramo:
edad_tramo
18–20    261
21–25    395

### **Sexo**

In [114]:
#@title
df2012['sexo'] = df2012['h3_sexo_del_entrevistado']
df2013['sexo'] = df2013['sexo_entrevistado']
df2015['sexo'] = df2015['sexo']
df2016['sexo'] = df2016['sexo_del_entrevistado']
df2017['sexo'] = df2017['sexo_entrevistado']
df2018['sexo'] = df2018['sexo_acceso']
df2024['sexo'] = df2024['a3_sexo_miembro_1']
#df2025['sexo'] = df2025['a3_sexo_miembro_1']

def estandarizar_sexo(df, columna='sexo'):
    df[columna] = (
        df[columna]
        .astype(str)
        .str.lower()
        .str.replace(r'\d+', '', regex=True)
        .str.replace(r'[^\w\s]', '', regex=True)
        .str.strip()
    )

    mapeo = {
        'mujer': 'mujer',
        'hombre': 'hombre'
    }

    df[columna] = df[columna].map(mapeo)

    return df

dfs = [df2012, df2013, df2015, df2016, df2017, df2018]

for df in dfs:
    estandarizar_sexo(df, 'sexo')

## **Variables tecnológicas**

### **Tipo de acceso a internet**

In [70]:
#@title
def _find_cols_contains(df, patterns, case=False):
    cols = []
    for c in df.columns:
        for p in patterns:
            if (p in c) if case else (p.lower() in c.lower()):
                cols.append(c)
                break
    return cols

def _to_yes(s):
    if s is None:
        return False
    if isinstance(s, (int, float)):
        return bool(int(s)) if not np.isnan(s) else False
    x = str(s).strip().lower()
    return x in ("1", "si", "sí", "s", "true", "x")

def _clean_str(x):
    if pd.isna(x):
        return np.nan
    return str(x).strip()

def armoniza_2012_2013(df, col_origen, blanks_to_ninguna=False):

    if col_origen not in df.columns:
        print(f"Warning: Column '{col_origen}' not found in dataframe. 'tipo_acceso' will not be created.")
        df["tipo_acceso"] = pd.NA
        return df

    s = df[col_origen].apply(_clean_str)

    if blanks_to_ninguna:
        s = s.fillna("ninguno").replace("", "ninguno")

    s_low = s.fillna("").str.lower()

    fija = s_low.str.contains(r"(adsl|cable|fibra|wifi|banda ancha fija)")
    movil = s_low.str.contains(r"(m[oó]vil|usb|celular|smartphone|tablet)")
    satelital = s_low.str.contains("satelital")
    otro = s_low.str.contains(r"(^otro|otro\.)")
    ninguna = s_low.str.contains("ningun")
    no_sabe = s_low.str.contains(r"(no sabe|ns/nr|no responde|no especifica)")

    df["tipo_acceso"] = np.select(
        [
            no_sabe,
            ninguna,
            fija & movil,
            fija,
            movil,
            satelital,
            otro
        ],
        [
            "No Sabe",
            "Ninguna",
            "Fija + Móvil",
            "Sólo Fija",
            "Sólo Móvil",
            "Sólo Satelital",
            "Otro"
        ],
        default="No Sabe"
    )

    return df

def armoniza_2016_2017(df, col_origen="conexion_hogares"):

    if col_origen not in df.columns:
        print(f"Warning: Column '{col_origen}' not found in dataframe. 'tipo_acceso' will not be created.")
        df["tipo_acceso"] = pd.NA
        return df

    s = df[col_origen].apply(_clean_str).fillna("No Sabe").str.lower()

    df["tipo_acceso"] = np.select(
        [
            s.str.contains("fija") & s.str.contains("m[oó]vil"),
            s.str.contains("solo fija|s[oó]lo fija"),
            s.str.contains("solo m[oó]vil|s[oó]lo m[oó]vil"),
            s.str.contains("ninguna|ninguno"),
            s.str.contains("no especifica|no sabe")
        ],
        [
            "Fija + Móvil",
            "Sólo Fija",
            "Sólo Móvil",
            "Ninguna",
            "No Sabe"
        ],
        default="No Sabe"
    )

    return df

def armoniza_multi_sino(df, year):

    fija_cols = _find_cols_contains(df, ["BANDA ANCHA FIJA", "WIFI", "FIJA"])
    movil_cols = _find_cols_contains(df, ["MÓVIL", "MOVIL", "USB", "SMARTPHONE", "TABLET"])
    sat_cols = _find_cols_contains(df, ["SATELITAL"])
    otro_cols = _find_cols_contains(df, ["OTRO"])
    ns_cols = _find_cols_contains(df, ["NO SABE", "NO ESPECIFICA", "NS"])

    def any_true(cols):
        if not cols:
            return pd.Series(False, index=df.index)
        return pd.DataFrame({c: df[c].map(_to_yes) for c in cols}).any(axis=1)

    fija = any_true(fija_cols)
    movil = any_true(movil_cols)
    satelital = any_true(sat_cols)
    otro = any_true(otro_cols)
    no_sabe = any_true(ns_cols)

    alguna = fija | movil | satelital | otro | no_sabe
    ninguna = ~alguna

    df["tipo_acceso"] = np.select(
        [
            no_sabe,
            ninguna,
            fija & movil,
            fija,
            movil,
            satelital,
            otro
        ],
        [
            "No Sabe",
            "Ninguna",
            "Fija + Móvil",
            "Sólo Fija",
            "Sólo Móvil",
            "Sólo Satelital",
            "Otro"
        ],
        default="No Sabe"
    )

    return df

df2012 = armoniza_2012_2013(df2012, "p7a_1_forma_de_acceso_a_internet")
df2013 = armoniza_2012_2013(df2013, "forma_de_acceso", blanks_to_ninguna=True)
df2015 = armoniza_multi_sino(df2015, 2015)
df2016 = armoniza_2016_2017(df2016)
df2017 = armoniza_2016_2017(df2017)
df2018 = armoniza_multi_sino(df2018, 2018)

for y, d in [("2012", df2012), ("2013", df2013), ("2015", df2015),
             ("2016", df2016), ("2017", df2017), ("2018", df2018)]:
    print(f"\n{y}")
    print(d["tipo_acceso"].value_counts())


2012
tipo_acceso
Ninguna           1722
Sólo Fija         1220
Sólo Móvil         672
Sólo Satelital       2
Otro                 1
Name: count, dtype: int64

2013
tipo_acceso
Sólo Fija         3531
Ninguna           3514
Sólo Móvil        1616
Otro                11
Sólo Satelital       8
No Sabe              2
Name: count, dtype: int64

2015
tipo_acceso
No Sabe           1165
Sólo Móvil         984
Fija + Móvil       848
Sólo Fija          232
Otro               231
Sólo Satelital     102
Name: count, dtype: int64

2016
tipo_acceso
Sólo Fija       1087
Ninguna         1086
Sólo Móvil       678
Fija + Móvil     584
No Sabe           36
Name: count, dtype: int64

2017
tipo_acceso
Sólo Fija       989
Fija + Móvil    903
Sólo Móvil      839
Ninguna         739
No Sabe          32
Name: count, dtype: int64

2018
tipo_acceso
No Sabe      3033
Ninguna       361
Sólo Fija      99
Otro           14
Name: count, dtype: int64


### **Uso de dispositivos (Computador y Smartphone)**

In [71]:
#@title
# uso de dispositivos
# 2012
df2012['usa_computador'] = np.where(df2012['p6_que_dispositivos_o_equipos_electronicos_utilizan_los_miembros_de_este_hogar'].isna(), 'no', 'si')
df2012['usa_smartphone'] = np.where(df2012['p6_que_dispositivos_o_equipos_electronicos_utilizan_los_miembros_de_este_hogar_1'].isna(), 'no', 'si')

# 2013
cols_computador = [
    "equipos_utilizados_para_acceder_a_internet_desde_el_hogar_computador_fijo",
    "equipos_utilizados_para_acceder_a_internet_desde_el_hogar_computador_portatil",
]

df2013[cols_computador] = (
    df2013[cols_computador]
    .replace(r"^\s*$", np.nan, regex=True)
)

df2013["usa_computador"] = np.where(
    df2013[cols_computador[0]].isna() & df2013[cols_computador[1]].isna(),
    "no",
    "si"
)

col_movil = (
    "equipos_utilizados_para_acceder_a_internet_desde_el_hogar_telefono_movil_o_sma"
)

df2013[col_movil] = df2013[col_movil].replace(r"^\s*$", np.nan, regex=True)

df2013["usa_smartphone"] = np.where(
    df2013[col_movil].isna(),
    "no",
    "si"
)

# 2015
df2015['usa_computador'] = np.where(
    df2015['computador_fijo_pc_cual_es_el_tipo_de_terminal_para_conectarse_a_internet_qu'].isna() &
    df2015['laptop_cual_es_el_tipo_de_terminal_para_conectarse_a_internet_que_usted_consid'].isna() &
    df2015['netbook_cual_es_el_tipo_de_terminal_para_conectarse_a_internet_que_usted_consi'].isna(),
    'no',
    'si'
)

df2015['usa_smartphone'] = np.where(
    df2015['smartphone_iphone_galaxy_palm_blackberry_xperia_etc_cual_es_el_tipo_de'].isna(),
    'no',
    'si'
)

# 2016
cols_computador16 = [
    "computador_fijo_pc_desktop_1",
    "computador_portatil_notebook_laptop_o_netbook_1",
]

df2016[cols_computador16] = (
    df2016[cols_computador16]
    .replace(r"^\s*$", np.nan, regex=True)
)

df2016["usa_computador"] = np.where(
    df2016[cols_computador16[0]].isna() & df2016[cols_computador16[1]].isna(),
    "no",
    "si"
)

col_movil16 = (
    "telefono_movil_o_smartphone_iphone_galaxy_palm_blackberry_xperia_etc"
)

df2016[col_movil16] = df2016[col_movil16].replace(r"^\s*$", np.nan, regex=True)

df2016["usa_smartphone"] = np.where(
    df2016[col_movil16].isna(),
    "no",
    "si"
)

# 2017
df2017['usa_computador'] = np.where(
    df2017['computador_fijo_pc_desktop'].isna() &
    df2017['computador_portatil_notebook_laptop_o_netbook'].isna(),
    'no',
    'si'
)

df2017["usa_smartphone"] = np.where(
    df2017["telefono_movil_o_smartphone_iphone_galaxy_palm_blackberry_xperia_etc"]
        .astype(str)
        .str.strip()
        .str.lower()
        .eq("yes"),
    "si",
    "no"
)

# 2018
df2018['usa_computador'] = np.where(
    df2018['p_6_2_cual_o_cuales_de_estos_dispositivos_o_equipos_utilizan_los_miembros_de_este_hogar_para_acceder_a_internet_desde_el_hogar'].isna() &
    df2018['p_6_2_cual_o_cuales_de_estos_dispositivos_o_equipos_utilizan_los_miembros_de_este_hogar_para_acceder_a_internet_desde_el_hogar_1'].isna(),
    'no',
    'si'
)

df2018['usa_smartphone'] = np.where(
    df2018['p_6_2_cual_o_cuales_de_estos_dispositivos_o_equipos_utilizan_los_miembros_de_este_hogar_para_acceder_a_internet_desde_el_hogar_3'].isna(),
    'no',
    'si'
)

# **Construcción del dataframe final**

In [72]:
#@title
def fusionar_dfs(dfs, annos, columnas=["anno", "region", "zona", "acceso_int", "quintil", "niveduc", "sexo", "edad", "tipo_acceso", "edad_tramo", "usa_computador", "usa_smartphone", "exp_hog"]):
    dfs_limpios = []

    for df, anno in zip(dfs, annos):
        df = df.copy()
        df["anno"] = anno
        df = df[columnas]
        dfs_limpios.append(df)

    return pd.concat(dfs_limpios, ignore_index=True)

dfs = [df2012, df2013, df2015, df2016, df2017, df2018]
annos = [2012, 2013, 2015, 2016, 2017, 2018]

df_final = fusionar_dfs(dfs, annos)


In [96]:
df2012_cl = df2012[["anno", "region", "zona", "acceso_int", "quintil", "niveduc", "sexo", "edad_tramo", "tipo_acceso", "usa_computador", "usa_smartphone", "exp_hog"]].copy()
df2013_cl = df2013[["anno", "region", "zona", "acceso_int", "quintil", "niveduc", "sexo", "edad_tramo", "tipo_acceso", "usa_computador", "usa_smartphone", "exp_hog"]].copy()
df2015_cl = df2015[["anno", "region", "zona", "acceso_int", "quintil", "niveduc", "sexo", "edad_tramo", "tipo_acceso", "usa_computador", "usa_smartphone", "exp_hog"]].copy()
df2016_cl = df2016[["anno", "region", "zona", "acceso_int", "quintil", "niveduc", "sexo", "edad_tramo", "tipo_acceso", "usa_computador", "usa_smartphone", "exp_hog"]].copy()
df2024_cl = df2024[["anno", "region", "zona", "acceso_int", "quintil", "niveduc", "sexo", "edad_tramo", "tipo_acceso", "usa_computador", "usa_smartphone", "exp_hog"]].copy()
df2025_cl = df2025[["anno", "region", "zona", "acceso_int", "quintil", "niveduc", "sexo", "edad_tramo", "tipo_acceso", "usa_computador", "usa_smartphone", "exp_hog"]].copy()

KeyError: "['quintil', 'niveduc', 'sexo', 'edad_tramo', 'tipo_acceso', 'usa_computador', 'usa_smartphone'] not in index"

In [95]:
df2018_cl['acceso_int'].value_counts()

acceso_int
si    3023
no     484
Name: count, dtype: int64

## **Análisis y limpieza de valores nulos**

In [73]:
#@title
# Limpieza de nulos y recodificación de 'quintil'

print("\n--- Análisis de nulos en df_final (antes) ---")
null_counts = df_final.isnull().sum()
cols_with_nulls = null_counts[null_counts > 0]

if not cols_with_nulls.empty:
    print(cols_with_nulls)
else:
    print("No hay valores nulos en df_final.")

df_final['quintil'] = df_final['quintil'].astype(str)
df_final['quintil'] = df_final['quintil'].replace(['99', '99.0'], pd.NA)

ordered_quintil_categories = ['Q1', 'Q2', 'Q3', 'Q4', 'Q5']
df_final['quintil'] = pd.Categorical(
    df_final['quintil'],
    categories=ordered_quintil_categories,
    ordered=True
)

print("\nValores '99' y '99.0' en 'quintil' reemplazados por NA.")

print("\n--- Análisis de nulos en df_final (después de recodificar quintil) ---")
null_counts = df_final.isnull().sum()
cols_with_nulls = null_counts[null_counts > 0]

if not cols_with_nulls.empty:
    print(cols_with_nulls)
else:
    print("No hay valores nulos en df_final.")

# Dimensiones antes y después de eliminar nulos
print(f"\nDimensiones antes de eliminar nulos: {df_final.shape}")

df_final = df_final.dropna().copy()

print(f"Dimensiones después de eliminar nulos: {df_final.shape}")



--- Análisis de nulos en df_final (antes) ---
quintil    1276
niveduc      24
dtype: int64

Valores '99' y '99.0' en 'quintil' reemplazados por NA.

--- Análisis de nulos en df_final (después de recodificar quintil) ---
quintil    2445
niveduc      24
dtype: int64

Dimensiones antes de eliminar nulos: (26341, 13)
Dimensiones después de eliminar nulos: (23882, 13)


## **Verificación de variables definitivas**

In [74]:
df_final.info()


<class 'pandas.core.frame.DataFrame'>
Index: 23882 entries, 0 to 26340
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   anno            23882 non-null  int64   
 1   region          23882 non-null  category
 2   zona            23882 non-null  object  
 3   acceso_int      23882 non-null  object  
 4   quintil         23882 non-null  category
 5   niveduc         23882 non-null  object  
 6   sexo            23882 non-null  object  
 7   edad            23882 non-null  float64 
 8   tipo_acceso     23882 non-null  object  
 9   edad_tramo      23882 non-null  object  
 10  usa_computador  23882 non-null  object  
 11  usa_smartphone  23882 non-null  object  
 12  exp_hog         23882 non-null  float64 
dtypes: category(2), float64(2), int64(1), object(8)
memory usage: 2.2+ MB


In [75]:
df_final.shape

(23882, 13)

In [76]:
df_final.head()

,anno,region,zona,acceso_int,quintil,niveduc,sexo,edad,tipo_acceso,edad_tramo,usa_computador,usa_smartphone,exp_hog
0,2012,10 los_lagos,rural,si,Q1,sup_universitaria_completa,mujer,60.00,Sólo Móvil,56–60,si,si,581.51
1,2012,11 aysen,rural,no,Q1,basica_incompleta,hombre,37.00,Ninguna,36–40,no,no,42.74
2,2012,11 aysen,rural,si,Q1,sup_universitaria_incompleta,hombre,41.00,Sólo Móvil,41–45,si,si,42.74
3,2012,11 aysen,rural,no,Q1,media_tp_incompleta,mujer,29.00,Ninguna,26–30,no,no,42.74
4,2012,11 aysen,rural,no,Q1,basica_completa,hombre,42.00,Ninguna,41–45,no,no,17.88


## **Exportación de DF a CSV para respaldo**

In [77]:
df_final.to_csv("df_final.csv", index=False)

# **Análisis exploratorio**

## **Frecuencias:** Variables *territoriales*

In [78]:
#@title
#print('===== REGION =====')
display(df_final.value_counts('region'))
#print(
#    'Nota: La región de Ñuble no figura con datos porque la edición 2018\n'
#    'de la encuesta se aplicó durante 2017, y la región fue creada el 5 de\n'
#    'septiembre de dicho año\n'
#)

#print('===== ZONA URBANA/RURAL =====')
display(df_final.value_counts('zona'))


region
13 metropolitana         4565
5 valparaiso             2694
8 biobio                 2416
2 antofagasta            1633
7 maule                  1533
9 araucania              1520
10 los_lagos             1437
6 ohiggins               1380
4 coquimbo               1314
14 los_rios              1186
11 aysen                  967
15 arica_y_parinacota     845
1 tarapaca                838
3 atacama                 827
12 magallanes             727
16 nuble                    0
Name: count, dtype: int64

zona
urbano    18940
rural      4942
Name: count, dtype: int64

## **Frecuencias:** Variables sociales, demográficas y económicas

In [79]:
#@title
print('\n===== SEXO =====')
display(df_final.value_counts('sexo'))

print('\n===== EDAD =====')
display(df_final.value_counts('edad'))

print('\n===== TRAMOS DE EDAD =====')
display(df_final.value_counts('edad_tramo'))

print('\n===== NIVEL EDUCACIONAL =====')
display(df_final.value_counts('niveduc'))

print('\n===== QUINTIL DE INGRESO =====')
display(df_final.value_counts('quintil'))


===== SEXO =====


sexo
mujer     13301
hombre    10581
Name: count, dtype: int64


===== EDAD =====


edad
30.00    575
18.00    567
50.00    551
40.00    546
45.00    531
        ... 
88.00      2
91.00      2
92.00      1
93.00      1
96.00      1
Name: count, Length: 77, dtype: int64


===== TRAMOS DE EDAD =====


edad_tramo
21–25    2557
26–30    2521
46–50    2251
36–40    2219
31–35    2196
41–45    2176
51–55    2122
56–60    2011
18–20    1516
61–65    1512
66–70    1430
71–75    1128
76–80     139
76+       104
Name: count, dtype: int64


===== NIVEL EDUCACIONAL =====


niveduc
media_ch_completa               5620
basica_incompleta               3068
media_tp_completa               2779
basica_completa                 2545
sup_universitaria_completa      2291
media_ch_incompleta             2102
sup_tecnica_completa            1898
sup_universitaria_incompleta    1639
media_tp_incompleta              968
sup_tecnica_incompleta           630
sin_educacion_formal             342
Name: count, dtype: int64


===== QUINTIL DE INGRESO =====


quintil
Q1    4944
Q2    4910
Q3    4891
Q5    4691
Q4    4446
Name: count, dtype: int64

## **Frecuencias:** Variables tecnológicas

In [80]:
#@title
print('\n===== ACCESO A INTERNET =====')
display(df_final.value_counts('acceso_int'))

print('\n===== TIPO DE ACCESO A INTERNET =====')
display(df_final.value_counts('tipo_acceso'))

print('\n===== USO DE COMPUTADOR =====')
display(df_final.value_counts('usa_computador'))

print('\n===== USO DE SMARTPHONE =====')
display(df_final.value_counts('usa_smartphone'))


===== ACCESO A INTERNET =====


acceso_int
si    15563
no     8319
Name: count, dtype: int64


===== TIPO DE ACCESO A INTERNET =====


tipo_acceso
Ninguna           6980
Sólo Fija         6585
Sólo Móvil        4577
No Sabe           3132
Fija + Móvil      2267
Otro               233
Sólo Satelital     108
Name: count, dtype: int64


===== USO DE COMPUTADOR =====


usa_computador
no    11970
si    11912
Name: count, dtype: int64


===== USO DE SMARTPHONE =====


usa_smartphone
no    14018
si     9864
Name: count, dtype: int64

# **Análisis**

In [89]:
print(df_final.columns)

Index(['anno', 'region', 'zona', 'acceso_int', 'quintil', 'niveduc', 'sexo',
       'edad', 'tipo_acceso', 'edad_tramo', 'usa_computador', 'usa_smartphone',
       'exp_hog'],
      dtype='object')


In [81]:
# FUNCION PARA ESTADÍSTICAS DESCRIPTIVAS PONDERADAS
def dstats(data_df, variables, tipo='frecuencia', variable_cruce=None, factor='exp_hog', transponer=False):
    """
    Realiza análisis ponderados para la ENUPE 2022.

    Parámetros:
    -----------
    data_df : DataFrame con los datos
    variables : str o lista de str con el nombre de la(s) variable(s)
    tipo : 'frecuencia', 'cruzada' o 'promedio'
    variable_cruce : str con la variable de cruce (solo para tipo='cruzada')
    factor : nombre de la columna del factor de expansión (default: 'exp')

    Ejemplos:
    ---------
    dstats(df, 'sexo_label', tipo='frecuencia')
    dstats(df, 'sexo_label', tipo='cruzada', variable_cruce='macrozona_label')
    dstats(df, ['edad', 'ingresos'], tipo='promedio')
    dstats(df, ['d1204', 'd1205'], tipo='promedio', variable_cruce='tamano', factor='fe_transversal')
    """
    if not isinstance(data_df, pd.DataFrame):
        return None

    # --- Validaciones ---
    if isinstance(variables, str):
        variables = [variables]

    columnas_necesarias = variables + [factor]
    if variable_cruce:
        columnas_necesarias.append(variable_cruce)

    for col_necesaria in columnas_necesarias:
        if col_necesaria not in data_df.columns:
            raise ValueError(f"La columna '{col_necesaria}' no existe en el DataFrame.")

    # --- Frecuencia ponderada ---
    if tipo == 'frecuencia':
        var = variables[0]
        total_pond = data_df[factor].sum()

        resultado = (
            data_df.groupby(var, observed=True)[factor]
            .sum()
            .reset_index()
            .rename(columns={factor: 'n_ponderado'})
        )
        resultado['porcentaje'] = (resultado['n_ponderado'] / total_pond * 100).round(2)
        resultado = resultado.sort_values('n_ponderado', ascending=False).reset_index(drop=True)

        fila_totales = pd.DataFrame({
            var: ['Total'],
            'n_ponderado': [resultado['n_ponderado'].sum()],
            'porcentaje': [resultado['porcentaje'].sum().round(2)]
        })

        resultado = pd.concat([resultado, fila_totales], ignore_index=True)
        return resultado

    # --- Tabla cruzada ponderada ---
    elif tipo == 'cruzada':
        if not variable_cruce:
            raise ValueError("Debes especificar 'variable_cruce' para tipo='cruzada'.")
        var = variables[0]

        tabla = data_df.pivot_table(
            values=factor,
            index=var,
            columns=variable_cruce,
            aggfunc='sum',
            fill_value=0,
            observed=False
        )

        tabla_pct = tabla.div(tabla.sum(axis=0), axis=1).mul(100).round(2)

        if transponer:
            tabla = tabla.T
            tabla_pct = tabla_pct.T

        tabla.loc['Total'] = tabla.sum(numeric_only=True)
        tabla_pct.loc['Total'] = tabla_pct.sum(numeric_only=True).round(2)

        display(tabla_pct)
        display(tabla)
        return tabla

    # --- Promedio ponderado ---
    elif tipo == 'promedio':

        # Sin cruce: comportamiento original
        if not variable_cruce:
            resultados = {}
            for var in variables:
                datos = data_df[[var, factor]].dropna()
                promedio = np.average(datos[var], weights=datos[factor])
                resultados[var] = float(round(promedio, 4))

            resultado_df = pd.DataFrame(
                list(resultados.items()),
                columns=['variable', 'promedio_ponderado']
            )
            return resultado_df

        # Con cruce: promedio ponderado por grupo
        else:
            filas = {}

            for grupo, subdf in data_df.groupby(variable_cruce, observed=True):
                fila = {}
                for var in variables:
                    datos = subdf[[var, factor]].dropna()
                    if len(datos) == 0 or datos[factor].sum() == 0:
                        fila[var] = np.nan
                    else:
                        fila[var] = float(round(np.average(datos[var], weights=datos[factor]), 4))
                filas[grupo] = fila

            # Fila Total sobre todo el df
            fila_total = {}
            for var in variables:
                datos = data_df[[var, factor]].dropna()
                fila_total[var] = float(round(np.average(datos[var], weights=datos[factor]), 4))
            filas['Total'] = fila_total

            resultado_df = pd.DataFrame(filas).T
            resultado_df.index.name = variable_cruce
            
            return resultado_df

        # --- Suma ponderada ---
    elif tipo == 'suma':

        # Sin cruce: suma ponderada total
        if not variable_cruce:
            resultados = {}
            for var in variables:
                datos = data_df[[var, factor]].dropna()
                # Cada valor se multiplica por su peso y se suman
                resultados[var] = float(round((datos[var] * datos[factor]).sum(), 4))

            resultado_df = pd.DataFrame(
                list(resultados.items()),
                columns=['variable', 'suma_ponderada']
            )
            return resultado_df

        # Con cruce: suma ponderada por grupo
        else:
            filas = {}

            for grupo, subdf in data_df.groupby(variable_cruce, observed=True):
                fila = {}
                for var in variables:
                    datos = subdf[[var, factor]].dropna()
                    fila[var] = float(round((datos[var] * datos[factor]).sum(), 4))
                filas[grupo] = fila

            # Fila Total sobre todo el df
            fila_total = {}
            for var in variables:
                datos = data_df[[var, factor]].dropna()
                fila_total[var] = float(round((datos[var] * datos[factor]).sum(), 4))
            filas['Total'] = fila_total

            resultado_df = pd.DataFrame(filas).T
            resultado_df.index.name = variable_cruce
            return resultado_df

    else:
        raise ValueError("El parámetro 'tipo' debe ser 'frecuencia', 'cruzada' o 'promedio'.")

In [88]:
dstats(df_final, 'region', tipo='frecuencia')


,region,n_ponderado,porcentaje
0,8 biobio,"88,109,772,526,440,416.00",61.79
1,15 arica_y_parinacota,"17,567,001,343,048,414.00",12.32
2,13 metropolitana,"10,870,073,560,340,930.00",7.62
3,5 valparaiso,"8,667,694,265,762,343.00",6.08
4,6 ohiggins,"7,673,850,220,922,986.00",5.38
5,9 araucania,"5,703,810,426,111,060.00",4.00
6,14 los_rios,"3,273,799,439,057,828.00",2.30
7,2 antofagasta,"726,799,889,368,158.88",0.51
8,7 maule,"1,491,332.85",0.00
9,10 los_lagos,"1,153,728.50",0.00


In [86]:
df_final['region'].value_counts(normalize=True)*100

region
13 metropolitana        19.11
5 valparaiso            11.28
8 biobio                10.12
2 antofagasta            6.84
7 maule                  6.42
9 araucania              6.36
10 los_lagos             6.02
6 ohiggins               5.78
4 coquimbo               5.50
14 los_rios              4.97
11 aysen                 4.05
15 arica_y_parinacota    3.54
1 tarapaca               3.51
3 atacama                3.46
12 magallanes            3.04
16 nuble                 0.00
Name: proportion, dtype: float64